# This file runs the drone simulation for all 3 cases:

In [ ]:
"""
pip install numpy, sympy, scipy, matplotlib
"""

import numpy as np
import sympy as sym
from scipy import linalg
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

# state: p (world frame), q=[qw,qx,qy,qz] (body to world frame), v (body frame), w (body frame)
p_x, p_y, p_z = sym.symbols('p_x p_y p_z')
qw, qx, qy, qz = sym.symbols('q_w q_x q_y q_z')
v_x, v_y, v_z, w_x, w_y, w_z = sym.symbols('v_x v_y v_z w_x w_y w_z')
tau_x, tau_y, tau_z, f_z = sym.symbols('tau_x tau_y tau_z f_z')
m_s, Jx_s, Jy_s, Jz_s, g_s = sym.symbols('m J_x J_y J_z g')

Rq = sym.Matrix([[1-2*(qy**2+qz**2), 2*(qx*qy-qw*qz),   2*(qx*qz+qw*qy)],
                 [2*(qx*qy+qw*qz),   1-2*(qx**2+qz**2), 2*(qy*qz-qw*qx)],
                 [2*(qx*qz-qw*qy),   2*(qy*qz+qw*qx),   1-2*(qx**2+qy**2)]])

v = sym.Matrix([v_x,v_y,v_z]) 
w = sym.Matrix([w_x,w_y,w_z])
J = sym.diag(Jx_s,Jy_s,Jz_s)

# time derivative of quaternion: q_dot = 1/2 q . [0,w]
q_dot = sym.Rational(1,2)*sym.Matrix([-qx*w_x-qy*w_y-qz*w_z,
                                       qw*w_x+qy*w_z-qz*w_y,
                                       qw*w_y-qx*w_z+qz*w_x,
                                       qw*w_z+qx*w_y-qy*w_x])

f_inB = Rq.T@sym.Matrix([0,0,-m_s*g_s]) + sym.Matrix([0,0,f_z])
tau = sym.Matrix([tau_x,tau_y,tau_z])
f_sym = sym.Matrix.vstack(Rq@v, # 3 velocity
                          q_dot, # 4 quaternion dot
                          (f_inB - w.cross(m_s*v))/m_s, # 3 acceleration
                          J.inv()@(tau - w.cross(J@w)))  # 3 angular acceleration (Euler's rigid body equation)
# 13 states total for EOM

S = [p_x,p_y,p_z,qw,qx,qy,qz,v_x,v_y,v_z,w_x,w_y,w_z]
U=[tau_x,tau_y,tau_z,f_z]
PR=[m_s,Jx_s,Jy_s,Jz_s,g_s]
f_num = sym.lambdify(S+U+PR, f_sym, 'numpy', cse=True)

# linearize
S12 = [p_x,p_y,p_z,qx,qy,qz,v_x,v_y,v_z,w_x,w_y,w_z] # 12 states by removing qw which is redundant and solvable
sub = {qw: sym.sqrt(1-qx**2-qy**2-qz**2)} # sub in for qw 
f12 = f_sym.subs(sub)
f12 = sym.Matrix([f12[i] for i in range(13) if i != 3])
A_sym = f12.jacobian(S12) # jacobian w.r.t state
B_sym = f12.jacobian(U) # jacobian w.r.t input
m,Jx,Jy,Jz,g = 0.528,0.003508,0.003468,0.006962,9.81 # calculated from components, # kg, kg m^2, kg m^2, kg m^2, m/s^2 respectively
par=[m,Jx,Jy,Jz,g]
eq = {s_:0 for s_ in S12}
eq.update({tau_x:0,tau_y:0,tau_z:0,f_z:m*g})
eq.update(dict(zip(PR,par)))
Aq = np.array(A_sym.subs(eq),float)
Bq = np.array(B_sym.subs(eq),float)

# small angle error quaternion, a = 2 * q_v
T = np.diag([1,1,1,2,2,2,1,1,1,1,1,1.]) # multiply attitude row by 2 for small angle rotation vector representation in A and B
A = T@Aq@np.linalg.inv(T)
B = T@Bq

# Controller
kF,kM,l = 3.95e-5,1.2e-7,0.159099025767 # coefficients of force and moment, l is the distance from motor to COM of frame from components
CMAX=65535 # configurable based on drone model selected, using 483 values for now
P = np.array([
    [-l*kF,-l*kF,l*kF,l*kF],
    [-l*kF,l*kF,l*kF,-l*kF],
    [-kM,kM,-kM,kM],
    [kF,kF,kF,kF]])
Pinv=linalg.inv(P)

x1_max = 0.02 # m, x
x2_max = 0.02 # m, y
x3_max = 0.05 # m, z
x4_max = np.deg2rad(5) # rad, psi
x5_max = np.deg2rad(5) # rad, theta
x6_max = np.deg2rad(5) # rad, phi
x7_max = 0.4 # m/s, vx
x8_max = 0.4 # m/s, vy
x9_max = 0.1 # m/s, vz
x10_max = np.deg2rad(55) # rad/s, wx
x11_max = np.deg2rad(55) # rad/s, wy
x12_max = np.deg2rad(65) # rad/s, wz

x = np.array([
    x1_max,
    x2_max,
    x3_max,
    x4_max,
    x5_max,
    x6_max,
    x7_max,
    x8_max,
    x9_max,
    x10_max,
    x11_max,
    x12_max
])

x2 = x**2

finalx = 1/x2

Q = np.diag(finalx)

c = CMAX - ((m*g) / (4 * kF))
delta = c / 4

u1_max = 4 * kF * l * delta
u2_max = 4 * kF * l * delta
u3_max = 4 * kM * delta
u4_max = 4 * kF * delta

R = np.diag([
    1 / u1_max**2,
    1 / u2_max**2,
    1 / u3_max**2,
    1 / u4_max**2
])

# Ricatti solution
Pm = linalg.solve_continuous_are(A,B,Q,R)
# gain K for LQR controller
K = linalg.inv(R)@B.T@Pm

print("closed-loop max real eigenvalues =", np.linalg.eigvals(A-B@K).real.max())


# Quaternion math helper functions
def qmul(a,b):
    return np.array([a[0]*b[0]-a[1]*b[1]-a[2]*b[2]-a[3]*b[3],
                     a[0]*b[1]+a[1]*b[0]+a[2]*b[3]-a[3]*b[2],
                     a[0]*b[2]-a[1]*b[3]+a[2]*b[0]+a[3]*b[1],
                     a[0]*b[3]+a[1]*b[2]-a[2]*b[1]+a[3]*b[0]])
def qconj(a): 
    return np.array([a[0],-a[1],-a[2],-a[3]])
def qyaw(psi): 
    return np.array([np.cos(psi/2),0,0,np.sin(psi/2)])
def Rot(q):
    w_,x,y,z=q
    return np.array([[1-2*(y*y+z*z),2*(x*y-w_*z),2*(x*z+w_*y)],[2*(x*y+w_*z),1-2*(x*x+z*z),2*(y*z-w_*x)],[2*(x*z-w_*y),2*(y*z+w_*x),1-2*(x*x+y*y)]])
def Rz_(a): 
    return np.array([[np.cos(a),-np.sin(a),0],[np.sin(a),np.cos(a),0],[0,0,1]])
def euler(q):   # 3-2-1 format
    w_,x,y,z=q.T
    psi=np.arctan2(2*(w_*z+x*y),1-2*(y*y+z*z))
    theta=np.arcsin(np.clip(2*(w_*y-z*x),-1,1))
    phi=np.arctan2(2*(w_*x+y*z),1-2*(x*x+y*y))
    return psi,theta,phi

# flights
# smooth velocity helper to prevent controller failure
def mj(t, t0, T_):
    u = np.clip((t - t0) / T_, 0, 1) # constant rate
    return u, (1. / T_ if 0 < (t - t0) < T_ else 0.) # fraction, rate

def ref_hover(t):
    return np.array([0,0,1.]),np.zeros(3),0.
def ref_circle(t):
    r,sp=2.,0.5
    om=sp/r
    return np.array([r*np.cos(om*t),r*np.sin(om*t),1.]),np.array([-sp*np.sin(om*t),sp*np.cos(om*t),0.]),0.
def ref_mission(t):
    pos=np.zeros(3)
    vel=np.zeros(3)
    s,sd=mj(t,0,5)
    pos[2]+=s
    vel[2]+=sd
    s,sd=mj(t,6,5)
    pos[0]+=5*s
    vel[0]+=5*sd
    yaw=np.pi/2*mj(t,13,3)[0]
    s,sd=mj(t,17,5)
    pos[1]+=5*s
    vel[1]+=5*sd
    s,sd=mj(t,24,190)
    pos[2]-=s
    vel[2]-=sd
    return pos,vel,yaw

# controller
def controller(s,ref,t):
    pd,vd,yd=ref(t)
    q=s[3:7]
    R=Rot(q)
    qe = qmul(qconj(qyaw(yd)),q) # attitude error (body frame)
    a = 2*np.sign(qe[0] if qe[0]!=0 else 1)*qe[1:]  # shortest-path small-angle vector accoutning for quaternion sign
    x = np.concatenate([Rz_(yd).T@(s[0:3]-pd), a, s[7:10]-R.T@vd, s[10:13]])
    u = -K@x + np.array([0,0,0,m*g])
    wm = np.clip(Pinv@u,0,CMAX)
    return P@wm, wm


# simulation
def simulate(ref,s0,Ttot,dt=0.005):
    n=int(Ttot/dt)
    t=np.arange(n+1)*dt
    Sx=np.zeros((n+1,13))
    Sx[0]=s0
    W=np.zeros((n+1,4))
    s=s0.copy()
    F=lambda s,u: np.array(f_num(*s,*u,*par),float).ravel()
    for k in range(n):
        u,wm=controller(s,ref,t[k])
        W[k]=wm
        k1=F(s,u)
        k2=F(s+dt/2*k1,u)
        k3=F(s+dt/2*k2,u)
        k4=F(s+dt*k3,u)
        s=s+dt/6*(k1+2*k2+2*k3+k4)
        s[3:7]/=np.linalg.norm(s[3:7]) # renormalise quaternion to prevent any error in definition
        if s[2]<0:
            s[2]=0
            R=Rot(s[3:7])
            vw=R@s[7:10]
            if vw[2]<0: 
                vw[2]=0
                s[7:10]=R.T@vw
        Sx[k+1]=s
    W[-1]=W[-2]
    return t,Sx,W

def traj3d(ax, Sx_plot, rp_plot, min_xy_span=1.0, z_floor=0.0, z_min_aspect=0.35):
    """Draw the 3D trajectory with sensible axis limits."""
    allp = np.vstack([Sx_plot[:, :3], rp_plot[:, :3]])
    lo, hi = allp.min(0), allp.max(0)

    def span(a, b, minspan):  # pad + enforce a minimum window
        c, h = (a + b) / 2, max((b - a) / 2 * 1.1, minspan / 2)
        return c - h, c + h
    xl, yl = span(lo[0], hi[0], min_xy_span), span(lo[1], hi[1], min_xy_span)
    zl = (z_floor, max(1.25 * hi[2], 0.5)) # full altitude range: ground to above max

    ax.plot(Sx_plot[:, 0], Sx_plot[:, 1], Sx_plot[:, 2], lw=1.8, label='actual trajectory')
    ax.plot(rp_plot[:, 0], rp_plot[:, 1], rp_plot[:, 2], 'k:', lw=1.5, label='reference trajectory')
    ax.plot(Sx_plot[:, 0], Sx_plot[:, 1], np.full(len(Sx_plot), zl[0]),     # ground shadow
            color='gray', lw=1, alpha=0.5, label='ground projection')
    ax.scatter(*Sx_plot[0, :3],  marker='o', s=40, color='green', label='start')
    ax.scatter(*Sx_plot[-1, :3], marker='X', s=50, color='red',   label='end')

    ax.set(xlim=xl, ylim=yl, zlim=zl,
           title='3D Position Trajectory',
           xlabel='X Position [m]', ylabel='Y Position [m]', zlabel='Z Position [m]')
    dx, dy, dz = xl[1]-xl[0], yl[1]-yl[0], zl[1]-zl[0]
    ax.set_box_aspect((dx, dy, max(dz, z_min_aspect * max(dx, dy))))   # keep z readable
    ax.view_init(elev=22, azim=-55)
    ax.legend(loc='upper left', fontsize=8, framealpha=0.85)

def plot(name, t, Sx, rp):
    # Round position data to 5 significant figures (plotting only)
    Sx_plot = Sx.copy()
    rp_plot = rp.copy()

    Sx_plot[:, :3] = np.vectorize(lambda x: float(f"{x:.5g}"))(Sx[:, :3])
    rp_plot[:, :3] = np.vectorize(lambda x: float(f"{x:.5g}"))(rp[:, :3])

    psi, th, ph = euler(Sx[:, 3:7])

    # Create subplots
    fig = plt.figure(figsize=(14, 9), constrained_layout=True)
    ax = np.empty((2, 2), dtype=object)

    ax[0, 0] = fig.add_subplot(221)
    ax[0, 1] = fig.add_subplot(222)
    ax[1, 0] = fig.add_subplot(223)
    ax[1, 1] = fig.add_subplot(224, projection='3d')

    # CoM position vs reference
    for i, n_ in enumerate('xyz'):
        line, = ax[0, 0].plot(t, Sx_plot[:, i], label=f'p_{n_}')
        ax[0, 0].plot(t, rp_plot[:, i], ':',
                      color=line.get_color(), lw=1)

    ax[0, 0].set(
        title='CoM Position (dotted = reference)',
        xlabel='t [s]',
        ylabel='Position [m]'
    )
    ax[0, 0].legend()

    # Euler angles
    for a_, n_ in zip(
        (psi, th, ph),
        ('psi (yaw)', 'theta (pitch)', 'phi (roll)')
    ):
        ax[0, 1].plot(t, np.rad2deg(a_), label=n_)

    ax[0, 1].set(
        title='Euler Angles',
        xlabel='t [s]',
        ylabel='Angle [deg]'
    )
    ax[0, 1].legend()

    # Quaternion components
    for i, n_ in enumerate(['q_w', 'q_x', 'q_y', 'q_z']):
        ax[1, 0].plot(t, Sx[:, 3 + i], label=n_)

    ax[1, 0].set(
        title='Quaternion Components',
        xlabel='t [s]'
    )
    ax[1, 0].legend()

    ## 3D trajectory (actual world-frame position)
    traj3d(ax[1, 1], Sx_plot, rp_plot)
    plt.show()

def world_v(Sx): 
    return np.array([Rot(r[3:7])@r[7:10] for r in Sx])
I4=[1,0,0,0]
"""
## Case 1:
s0=np.array([0.0,-0.0,0.0,*I4,0,0,0,0,0,0])
t,Sx,W=simulate(ref_hover,s0,120)
rp=np.array([ref_hover(x)[0] for x in t])
_,th,ph=euler(Sx[:,3:7])
print("test 1 max err after 10 s:",np.abs(Sx[t>10,:3]-rp[t>10]).max(0)," max tilt deg",np.rad2deg(max(abs(th).max(),abs(ph).max())))
plot('flight1_hover',t,Sx,rp)


## Case 2:
s0=np.array([2,0,1,*I4,0,0.5,0,0,0,0])
t,Sx,W=simulate(ref_circle,s0,70)
rp=np.array([ref_circle(x)[0] for x in t])
vw=world_v(Sx)
r=np.hypot(Sx[:,0],Sx[:,1])
sp=np.hypot(vw[:,0],vw[:,1])
k=t>10
print("test 2 radius",r[k].min(),r[k].max()," speed",sp[k].min(),sp[k].max()," z",Sx[k,2].min(),Sx[k,2].max())
plot('flight2_circle',t,Sx,rp)

### Case 3
s0=np.array([0,0,0,*I4,0,0,0,0,0,0])
t,Sx,W=simulate(ref_mission,s0,215)
rp=np.array([ref_mission(x)[0] for x in t])
vw=world_v(Sx)
psi,_,_=euler(Sx[:,3:7])
k=int(24/.005)
print("test 3 final pos",Sx[-1,:3]," yaw@17s deg",np.rad2deg(psi[int(17/.005)])," peak descent speed",np.abs(vw[k:,2]).max(),"vz at ground",vw[-1,2]," min z",Sx[:,2].min())
print("max pos err",np.abs(Sx[:,:3]-rp).max(0)," motor cmd range",W.min(),W.max())
plot('flight3_mission',t,Sx,rp)"""

print(K)

closed-loop max real eigenvalues = -1.8156077291613841
[[-1.36745260e-15 -1.02914199e+01  5.73352790e-16  4.74490345e+00
  -1.81639673e-16 -1.74731496e-14  3.77612786e-15 -3.19691990e+00
   2.81024399e-16  2.81542479e-01  4.20340220e-17 -1.03213626e-14]
 [ 1.02914199e+01  1.50309637e-14  3.12826764e-15 -3.16446636e-15
   4.73993990e+00 -3.76226442e-15  3.19529069e+00  1.96678690e-15
   4.19876512e-16  4.25188434e-17  2.80806247e-01 -1.71648158e-15]
 [-1.25634493e-17  5.36151544e-16  1.54117652e-15 -1.12185159e-16
  -1.78645111e-18  4.50375288e-02 -3.13130666e-18  1.70783095e-16
   7.77809760e-17 -1.89625118e-18 -3.11757907e-19  2.52805220e-02]
 [ 2.90356975e-14 -1.39353260e-14  2.58742500e+01  7.40683974e-15
   6.51779333e-15  1.15551474e-13  1.28220502e-14 -7.37957159e-15
   1.39532223e+01  7.37623338e-17  1.08951122e-16  1.11123314e-13]]
